# Mouse Brain Spatial Gallery (Model-based)

This notebook loads the Mouse Brain Spatial Multiome dataset and each trained model, then
computes Leiden clusters from the topic representation and plots them in spatial coordinates.


In [ ]:
from pathlib import Path
import numpy as np
import scanpy as sc
import mudata as md
import scipy.sparse as sp
import matplotlib.pyplot as plt
from omics_topic.models.amortizedLDA import MultimodalAmortizedLDA


def compute_moran_i(values, spatial_weights):
    """Compute Moran's I statistic for a 1D array of values."""
    n = len(values)
    z = values - values.mean()
    W = spatial_weights.tocsr()
    numerator = z @ W @ z
    denominator = z @ z
    if denominator == 0:
        return 0.0
    S0 = W.sum()
    if S0 == 0:
        return 0.0
    return (n / S0) * (numerator / denominator)


# ---- Paths ----
DATA_DIR = Path("/data/Data_SpatialGlue/Dataset10_Mouse_Brain_H3K27me3")
MODEL_BASE_DIRS = [
    #Path("/data/omics_topic_models/mouse_brain_spatial"),
    Path("/data/omics_topic_models/mouse_brain_spatial_graphconv"),
    #Path("/data/omics_topic_models/mouse_brain_spatial_gcn_2"),
]

# ---- Load dataset (same preprocessing as training) ----
adata_atac = sc.read_h5ad(DATA_DIR / "adata_peaks_normalized.h5ad")
adata_rna = sc.read_h5ad(DATA_DIR / "adata_RNA.h5ad")

X = adata_atac.X
if sp.issparse(X):
    X = X.tocsr(copy=True)
    X.data = np.ones_like(X.data)
    X.eliminate_zeros()
    adata_atac.layers["binary"] = X
else:
    adata_atac.layers["binary"] = (X != 0).astype(np.float32)

mdata = md.MuData({"rna": adata_rna, "atac": adata_atac})

# Build spatial neighbor graph (for completeness; not used in plotting)
sc.pp.neighbors(
    mdata.mod["rna"],
    use_rep="spatial",
    n_neighbors=5,
    metric="euclidean",
    key_added="spatial",
)

mdata.obsp["spatial_connectivities"] = mdata.mod["rna"].obsp["spatial_connectivities"]
mdata.mod["atac"].obsp["spatial_connectivities"] = mdata.mod["rna"].obsp["spatial_connectivities"]
mdata.mod["atac"].obsp["spatial_distances"] = mdata.mod["rna"].obsp["spatial_distances"]

# Spatial coordinates
if "spatial" in mdata.mod["rna"].obsm:
    spatial_coords = mdata.mod["rna"].obsm["spatial"]
elif "spatial" in mdata.obsm:
    spatial_coords = mdata.obsm["spatial"]
else:
    raise ValueError("No spatial coordinates found in obsm")

spatial_weights = mdata.obsp["spatial_connectivities"]

# ---- Collect model runs ----
run_dirs = set()
for base in MODEL_BASE_DIRS:
    if not base.exists():
        continue
    for latent in base.rglob("latent_representation.npy"):
        run_dirs.add(latent.parent)
    for model_dir in base.rglob("model"):
        if model_dir.is_dir():
            run_dirs.add(model_dir.parent)

run_dirs = sorted(run_dirs)
print(f"Found {len(run_dirs)} model runs")

# ---- Plot per model ----
for run_dir in run_dirs:
    print(f"=== {run_dir} ===")

    theta = None
    latent_path = run_dir / "latent_representation.npy"
    if latent_path.exists():
        theta = np.load(latent_path)
    else:
        model_path = run_dir / "model"
        if model_path.exists():
            try:
                model = MultimodalAmortizedLDA.load(str(model_path), adata=mdata)
                theta = model.get_latent_representation(batch_size=mdata.n_obs).values
            except Exception as e:
                print(f"  Skipping (model load failed): {e}")
                continue
        else:
            print("  Skipping (no latent or model found)")
            continue

    # Compute mean Moran's I across latent dimensions
    morans_values = [compute_moran_i(theta[:, i], spatial_weights) for i in range(theta.shape[1])]
    mean_morans = np.mean(morans_values)

    # Topic representation -> Leiden
    mdata.mod["rna"].obsm["X_topic"] = theta - 1
    sc.pp.neighbors(mdata.mod["rna"], metric="cosine", use_rep="X_topic", key_added="topic_neighbors")
    sc.tl.leiden(mdata.mod["rna"], neighbors_key="topic_neighbors", key_added="leiden")

    leiden = mdata.mod["rna"].obs["leiden"].astype("category")
    leiden_encoded = leiden.cat.codes.to_numpy()

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(
        spatial_coords[:, 0],
        spatial_coords[:, 1],
        c=leiden_encoded,
        cmap="tab20",
        s=8,
        alpha=0.8,
        linewidths=0,
    )
    ax.set_title(f"{run_dir.name} | Moran's I = {mean_morans:.4f}")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_aspect("equal")
    ax.invert_yaxis()
    plt.show()
